# AtaRehber: evaluation on Google Colab

Runs `python -m evals.run_eval` with Ollama on a Colab GPU, once per model of `MODELS` and per entry of
`VARIANTS`, saves the reports to Google Drive, and compares them in the last cell.

**Before running:**
* **Runtime → Change runtime type → GPU:** T4 (free, 15 GB) runs `qwen3:8b` (about 5.2 GB) with an 8192-token
  context; L4 or A100 (Colab Pro) are faster.
* **Secrets** (key icon on the left): `GITHUB_TOKEN` with read access to the repository, *Notebook access* on.

In [ ]:
# Settings
REPO = "https://github.com/SirAlper/AtaRehber.git"
BRANCH = "main"
# Models to measure (Ollama names). qwen3:8b is about 5.2 GB; qwen2.5:14b (about 9 GB) also fits a T4.
MODELS = ["qwen3:8b"]         # e.g. ["qwen3:8b", "qwen2.5:14b"]
# Settings to compare (names of .env.example). Every model runs once per entry; {} is the default behaviour
# (qwen3 thinks for answers, compliance verdicts, and combined answers: OLLAMA_THINK_TASKS).
VARIANTS = [{}]               # e.g. [{}, {"OLLAMA_THINK_TASKS": "none"}, {"OLLAMA_THINK_TASKS": "answer,compliance,synthesis,grader"}]
GRADER_MODEL = ""             # model that checks the answers, e.g. "qwen2.5:7b"; empty: each model checks its own
# Context window in tokens (the default). Contexts with the neighbouring pieces of an article and the rules of a
# compliance scenario, plus qwen3's thinking, need more than 4096; a prompt that does not fit loses its beginning
# without an error.
NUM_CTX = 8192
DATASET = "university"        # "university" (127 questions, laws and regulations) or "demo" (51 questions)
STAGES = "all"                # "retrieval", "retrieval,routing", "all", "agents", "grader"
IDS = ""                      # e.g. "u-rek-01,u-rek-02" for a quick trial; empty: every question
# Reports are saved here (Google Drive survives the session); empty keeps them in /content/results
DRIVE_DIR = "/content/drive/MyDrive/AtaRehber"

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print("WARNING: the project is tested on Python 3.12+; this image is older and may fail.")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: choose Runtime > Change runtime type > GPU"

## 1. Code and Python packages

PyTorch is not installed: the image's own build (with CUDA) is used.

In [ ]:
%cd /content
from google.colab import userdata
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
clone_url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
!rm -rf AtaRehber && git clone --quiet --depth 1 --branch {BRANCH} {clone_url} AtaRehber
%cd /content/AtaRehber
# The token is not kept in the clone
!git remote set-url origin {REPO}
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Ollama and the language models

In [ ]:
import subprocess, time, urllib.request

# zstd unpacks the Ollama download; pciutils lets the installer see the GPU
!apt-get install -y -qq zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /content/ollama.log")
for model in dict.fromkeys(MODELS + ([GRADER_MODEL] if GRADER_MODEL else [])):
    print("Pulling", model)
    !ollama pull {model} 2>&1 | tail -1

## 3. Embedding and reranker models (about 3.3 GB)

In [ ]:
!python download_model.py 2>&1 | tail -5

## 4. Evaluation

Each question prints a line when it starts and one when it is done, with the progress and the time left. The
runs follow one another; `ollama ps` after each run shows whether the model was fully on the GPU.

In [ ]:
import os

if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
results = f"{DRIVE_DIR}/results" if DRIVE_DIR else "/content/results"
os.makedirs(results, exist_ok=True)

base_env = dict(os.environ)
args = f"--stages {STAGES} --out {results}"
if DATASET == "university":
    args += " --dataset evals/dataset_university.jsonl --corpus evals/corpus_university --no-database"
if IDS:
    args += f" --ids {IDS}"
runs = [(model, variant) for model in MODELS for variant in VARIANTS]
# Models added to the settings after section 2 ran are pulled here (a model already there takes seconds), and
# the router and grader models a variant names
named = [v[key] for v in VARIANTS for key in ("OLLAMA_ROUTER_MODEL", "OLLAMA_GRADER_MODEL") if v.get(key)]
for model in dict.fromkeys(MODELS + ([GRADER_MODEL] if GRADER_MODEL else []) + named):
    !ollama pull {model} 2>&1 | tail -1
for number, (model, variant) in enumerate(runs, 1):
    os.environ.clear()
    os.environ.update(base_env)
    os.environ.update({"OLLAMA_MODEL": model, "OLLAMA_GRADER_MODEL": GRADER_MODEL, "RAG_DEVICE": "cuda"})
    os.environ["OLLAMA_NUM_CTX"] = str(NUM_CTX)
    if DATASET == "university":
        # The threshold measured for article-aware chunks (docs/evaluation.md)
        os.environ["RAG_MIN_RERANKER_SCORE"] = "0.05"
    os.environ.update({key: str(value) for key, value in variant.items()})
    print(f"\n######## Run {number}/{len(runs)}: {model} {variant or '(default settings)'} ########")
    print("python -m evals.run_eval", args)
    !python -m evals.run_eval {args} 2>&1 | grep -v -E "^\[20|Batches|Loading weights"
    !ollama ps
    if model != GRADER_MODEL and (number == len(runs) or runs[number][0] != model):
        # Free the GPU memory for the next model
        !ollama stop {model}

## 5. The reports side by side

In [ ]:
import glob, json

paths = sorted(glob.glob(f"{results}/report_*.json"))[-len(runs):]
if not paths:
    raise SystemExit("No report was written; see the output above.")
reports = [json.load(open(path, encoding="utf-8")) for path in paths]
labels = [f"run {number}" for number in range(1, len(reports) + 1)]
for label, path, report, (model, variant) in zip(labels, paths, reports, runs[-len(reports):]):
    settings = report["config"]
    print(f"{label}: {settings['llm_model']} {variant or '(default settings)'}"
          f" | thinking for: {', '.join(settings.get('ollama_think_tasks') or []) or 'none'}")
    print(f"   checked by {settings.get('grader_model', settings['llm_model'])} | commit {report['git_commit']}"
          f" | {report['case_count']} cases | {path}")

width = 12
for stage in reports[0]["summary"]:
    print(f"\n== {stage} ==")
    print(" " * 36 + "".join(f"{label:>{width}}" for label in labels))
    for key, value in reports[0]["summary"][stage].items():
        if isinstance(value, (dict, list)):
            continue
        values = [report["summary"].get(stage, {}).get(key) for report in reports]
        shown = [f"{v:.3f}" if isinstance(v, float) else str(v) for v in values]
        print(f"  {key:<34}" + "".join(f"{text:>{width}}" for text in shown))